# Data Cleaning

## Load and Clean Price Data

In [23]:
import pandas as pd
from pathlib import Path

In [5]:
df_prices = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Raw\day_ahead_prices.parquet"
)

In [7]:
print(df_prices.head())
print(df_prices.info())

                           day_ahead_price
2022-01-01 00:00:00+01:00            50.05
2022-01-01 01:00:00+01:00            41.33
2022-01-01 02:00:00+01:00            43.22
2022-01-01 03:00:00+01:00            45.46
2022-01-01 04:00:00+01:00            37.67
<class 'pandas.DataFrame'>
DatetimeIndex: 26278 entries, 2022-01-01 00:00:00+01:00 to 2024-12-30 23:00:00+01:00
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   day_ahead_price  26278 non-null  float64
dtypes: float64(1)
memory usage: 410.6 KB
None


In [9]:
# Reindexing Price Data
full_range = pd.date_range(
    start=df_prices.index.min(),
    end=df_prices.index.max(),
    freq="h",
    tz="Europe/Berlin"
)

df_prices = df_prices.reindex(full_range)

In [11]:
# interpolate missing values
df_prices = df_prices.interpolate(method="time")

In [13]:
# Time zone to UTC
df_prices = df_prices.tz_convert("UTC")

In [25]:
# Saving cleaned data
processed_path = Path(r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed")

processed_path.mkdir(parents=True, exist_ok=True)

df_prices.to_parquet(
    processed_path / r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed\day_ahead_prices_cleaned.parquet"
)

In [27]:
print(df_prices.info())
print(df_prices.head())

<class 'pandas.DataFrame'>
DatetimeIndex: 26280 entries, 2021-12-31 23:00:00+00:00 to 2024-12-30 22:00:00+00:00
Freq: h
Data columns (total 1 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   day_ahead_price  26280 non-null  float64
dtypes: float64(1)
memory usage: 410.6 KB
None
                           day_ahead_price
2021-12-31 23:00:00+00:00            50.05
2022-01-01 00:00:00+00:00            41.33
2022-01-01 01:00:00+00:00            43.22
2022-01-01 02:00:00+00:00            45.46
2022-01-01 03:00:00+00:00            37.67


### Load and Clean 'Load' data

In [29]:
df_load = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Raw\load_raw.parquet"
)

In [31]:
# Reindexing Load Data
full_range = pd.date_range(
    start=df_load.index.min(),
    end=df_load.index.max(),
    freq="h",
    tz="Europe/Berlin"
)

df_load = df_load.reindex(full_range)

In [33]:
# interpolate missing values
df_load = df_load.interpolate(method="time")

In [35]:
# Time zone to UTC
df_load = df_load.tz_convert("UTC")

In [71]:
df_load = df_load.resample("h").mean()

In [73]:
# Saving cleaned data
processed_path = Path(r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed")

processed_path.mkdir(parents=True, exist_ok=True)

df_load.to_parquet(
    processed_path / r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Pre-Processed\load_cleaned.parquet"
)

### Load and Clean Renewables data

In [45]:
df_generation = pd.read_parquet(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\Raw\generation_raw.parquet"
)

In [49]:
df_generation = df_generation[
    [
        ("Solar", "Actual Aggregated"),
        ("Wind Offshore", "Actual Aggregated"),
        ("Wind Onshore", "Actual Aggregated")
    ]
]

In [51]:
df_generation.columns = [
    "solar_generation",
    "wind_offshore",
    "wind_onshore"
]

In [53]:
df_generation["wind_generation"] = (
    df_generation["wind_offshore"] +
    df_generation["wind_onshore"]
)

In [55]:
df_generation = df_generation[
    [
        "solar_generation",
        "wind_generation"
    ]
]

In [57]:
print(df_generation.info())
print(df_generation.head())

<class 'pandas.DataFrame'>
DatetimeIndex: 105118 entries, 2022-01-01 00:00:00+01:00 to 2024-12-30 23:45:00+01:00
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   solar_generation  105118 non-null  float64
 1   wind_generation   105118 non-null  float64
dtypes: float64(2)
memory usage: 2.4 MB
None
                           solar_generation  wind_generation
2022-01-01 00:00:00+01:00              1.91         31947.67
2022-01-01 00:15:00+01:00              1.68         31810.16
2022-01-01 00:30:00+01:00              1.70         31349.86
2022-01-01 00:45:00+01:00              1.62         31205.24
2022-01-01 01:00:00+01:00              1.70         30581.35


In [59]:
df_generation_hourly = (
    df_generation
    .resample("h")
    .mean()
)

In [61]:
# Reindexing Load Data
full_range = pd.date_range(
    start=df_generation_hourly.index.min(),
    end=df_generation_hourly.index.max(),
    freq="h",
    tz="Europe/Berlin"
)

df_generation_hourly = (
    df_generation_hourly
    .reindex(full_range)
)

In [63]:
# interpolate missing values
df_generation_hourly = (
    df_generation_hourly
    .interpolate(method="time")
)

In [65]:
# Time zone to UTC
df_generation_hourly = (
    df_generation_hourly
    .tz_convert("UTC")
)

In [69]:
print(df_generation_hourly.info())
print(df_generation_hourly.head())

<class 'pandas.DataFrame'>
DatetimeIndex: 26280 entries, 2021-12-31 23:00:00+00:00 to 2024-12-30 22:00:00+00:00
Freq: h
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   solar_generation  26280 non-null  float64
 1   wind_generation   26280 non-null  float64
dtypes: float64(2)
memory usage: 615.9 KB
None
                           solar_generation  wind_generation
2021-12-31 23:00:00+00:00            1.7275       31578.2325
2022-01-01 00:00:00+00:00            1.7450       30123.1350
2022-01-01 01:00:00+00:00            1.7700       28737.1700
2022-01-01 02:00:00+00:00            1.7950       27496.2600
2022-01-01 03:00:00+00:00            1.7700       26207.4875


In [76]:
# Saving cleaned data
save_path = Path(
    r"C:\Users\SherM\OneDrive\Documents\1. Thesis\Dataset\pre-processed"
)

save_path.mkdir(parents=True, exist_ok=True)

df_generation_hourly.to_parquet(
    save_path / "generation_cleaned.parquet"
)

print("Generation data saved successfully.")

Generation data saved successfully.
